# Exp 3 - Throughput and Response-Time Analysis in Autonomous Communication

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Estimate throughput and response time for autonomous communication packets.

Throughput tells how much payload is delivered per second. Response time tells how long each message takes from generation to completion. A communication link can have high throughput but still be unsuitable for control if response time is too large or too variable.

## Core Real-Time Systems Theory Notes

### 1. Introduction to Real-Time Systems

A real-time system is a computing system in which correctness depends on two things: the logical correctness of the output and the time at which the output is produced. In a normal general-purpose system, a late answer may be inconvenient. In a real-time system, a late answer may be useless or may cause unsafe behavior.

Logical correctness means that the calculated value or decision is correct. Temporal correctness means that the value or decision is available within the required time bound. A vehicle braking controller, robotic arm controller, industrial motor drive, medical monitoring device, avionics controller, or power-grid protection unit must satisfy both.

Real-time does not simply mean "fast." A fast system that sometimes misses its deadline is not dependable for hard real-time control. A slower system with bounded and predictable timing may be more suitable if it always meets the required deadline. The key engineering properties are determinism, predictability, bounded latency, and analyzable worst-case behavior.

General-purpose systems optimize average response, throughput, fairness, and user convenience. Real-time systems optimize deadline satisfaction, bounded response time, and predictable behavior under defined load. This is why real-time operating systems, embedded controllers, field buses, and deterministic networks often use priority policies, static configuration, time slots, or admission control.

### 2. Classification of Real-Time Systems

Hard real-time systems must not miss deadlines. A missed deadline is treated as system failure. Examples include autonomous emergency braking, airbag control, flight-control surfaces, pacemaker control, and industrial safety shutdown.

Firm real-time systems can tolerate some missed deadlines, but a late result has no value and is discarded. Examples include object-detection frames that arrive after the object is no longer relevant, traffic-sign recognition after the vehicle has passed the sign, or a stale cooperative-awareness message in V2X communication.

Soft real-time systems tolerate deadline misses with quality degradation. Examples include dashboard display refresh, infotainment audio buffering, non-critical telemetry upload, passenger comfort control, and route-estimation updates.

The classification depends on the consequence of lateness, not only the application name. A camera pipeline may be hard real-time when used for emergency braking, firm real-time when used for immediate object tracking, and soft real-time when used for driver display recording.

### 3. Real-Time Tasks and Events

A task is a schedulable unit of computation. In autonomous systems, tasks may represent sensor sampling, frame processing, message transmission, controller update, actuator command generation, logging, or security checking.

Periodic tasks occur at fixed intervals. Example: sample wheel speed every 10 ms. A periodic task is commonly described by execution time C, period T, and deadline D.

Aperiodic tasks occur irregularly and do not have a guaranteed minimum inter-arrival time. Example: a user opens a diagnostic screen. Aperiodic work is often lower criticality or handled by background servers.

Sporadic tasks occur irregularly but have a known minimum separation between arrivals. Example: emergency obstacle events may occur unpredictably but cannot arrive faster than a defined physical or system limit. Sporadic modelling is useful because it allows worst-case analysis.

Time-triggered events are released by a clock schedule. They improve predictability because activation times are known in advance. Event-triggered events are released when an external condition occurs, such as receiving a packet, detecting an obstacle, or crossing a threshold. Event-triggered systems are responsive but require careful overload handling.

### 4. Timing Parameters

The event occurrence time is the real-world time at which the physical event occurs. Release time is when the corresponding task becomes ready for scheduling. Arrival time is often used for the time at which a job enters a queue or a packet reaches a node. Start time is when execution actually begins. Execution time or computation time is the CPU or processor time consumed by the job.

Waiting time is the time spent ready but not executing:

```
waiting_time = start_time - release_time
```

Completion time or finish time is when the job finishes. Response time is the delay from release or arrival to completion:

```
response_time = finish_time - release_time
```

In many lab contexts, turnaround time is also:

```
turnaround_time = finish_time - arrival_time
```

If release time and arrival time are the same, response time and turnaround time become numerically equal. In networked systems they may differ because a real-world event can occur before the software task is released, or a packet can be generated before it reaches the receiving queue.

### 5. Timing Constraints

A relative deadline is measured from release time. An absolute deadline is a time on the system timeline:

```
absolute_deadline = release_time + relative_deadline
```

A deadline is met when:

```
finish_time <= absolute_deadline
```

A deadline miss occurs when:

```
finish_time > absolute_deadline
```

Deadline margin shows how much time remains at completion:

```
deadline_margin = absolute_deadline - finish_time
```

Positive margin means the task finished early. Zero means it finished exactly at the deadline. Negative margin means a miss.

Slack time estimates available spare time before a deadline:

```
slack = absolute_deadline - current_time - remaining_execution_time
```

Laxity is often used similarly:

```
laxity = deadline - current_time - remaining_computation_time
```

Worst-Case Execution Time, or WCET, is the maximum execution time under defined assumptions. Best-Case Execution Time, or BCET, is the minimum. Average execution time is not enough for hard real-time certification because rare long execution paths still matter.

### 6. Communication Performance Parameters

Latency is the time taken for data to move from source to destination:

```
latency = receive_time - send_time
```

Jitter is variation in latency. A simple packet-to-packet jitter estimate is:

```
jitter_i = abs(latency_i - latency_(i-1))
```

Throughput is useful delivered data per unit time:

```
throughput = delivered_bits / observation_time
```

Bandwidth is the nominal or available capacity of a link. Throughput is what is actually achieved after overhead, contention, retransmission, protocol limits, and congestion.

Packet transmission time is:

```
transmission_time = packet_size_bits / link_rate_bits_per_second
```

End-to-end delay can be modeled as:

```
end_to_end_delay = processing_delay + queueing_delay + transmission_delay + propagation_delay
```

Communication overhead is the extra data or time consumed by headers, acknowledgements, encryption, retransmission, routing, and synchronization. Packet loss affects reliability:

```
packet_loss_rate = lost_packets / sent_packets
reliability = delivered_packets / sent_packets
```

### 7. Real-Time Communication Requirements

Bounded latency means there is a known upper limit for message delay under defined conditions. Low jitter means delay stays stable across transmissions. Predictable communication means the designer can reason about message timing before deployment. Reliability means messages are delivered with acceptable probability or with recovery mechanisms. Availability means the communication service is usable when needed.

Deterministic message delivery is often achieved through priority arbitration, time slots, traffic shaping, redundancy, admission control, or real-time Ethernet features. Deadline-aware communication means messages are scheduled according to urgency and usefulness, not simply first-come first-served.

### 8. Timing Analysis in Autonomous Systems

A typical autonomous timing chain is:

```
Sensor -> Perception -> Planning/Control -> Actuator -> Physical Response
```

The perception-to-action delay is:

```
perception_to_action_delay =
    sensor_capture_time
  + sensor_preprocessing_time
  + perception_inference_time
  + planning_time
  + control_time
  + communication_time
  + actuator_response_time
```

For an autonomous braking example:

```
stopping_distance = reaction_distance + braking_distance
reaction_distance = vehicle_speed * total_system_delay
braking_distance = vehicle_speed^2 / (2 * deceleration)
```

Deadline verification compares the computed or measured response time with the maximum safe response time:

```
system_is_timely = measured_response_time <= required_deadline
```

Case Study - Autonomous Emergency Braking:
A front sensor detects an obstacle at a fixed distance. The system must capture sensor data, process it, decide, transmit the command, and apply braking before the remaining stopping distance becomes unsafe. The case study shows why real-time correctness is a chain property. A fast perception algorithm alone is not enough if the actuator command is delayed.

Case Study - Robotic Arm in Industrial Automation:
A robotic arm must stop when a worker crosses a safety boundary. Sensor detection, controller scheduling, network delivery, and motor-drive response must all be bounded. High average throughput is irrelevant if one delayed safety packet allows the arm to continue moving too long.

Case Study - V2X Hazard Warning:
A vehicle broadcasts a hazard message to nearby vehicles. The message is useful only if received before the receiving vehicle must react. This connects communication latency, jitter, packet loss, message freshness, and security verification.

### 9. Textbook Design Workflow for Real-Time Experiments

When solving a real-time lab problem, use a disciplined workflow. First identify the physical event or communication event. Second identify the software task or network message created by that event. Third list the timing parameters: release time, start time, execution time, finish time, and deadline. Fourth compute the response time and deadline margin. Fifth classify the consequence of lateness as hard, firm, or soft. Sixth propose a design improvement if the deadline is missed.

For autonomous systems, the timing boundary should be tied to a physical reason. For example, a braking deadline should relate to speed, distance, and deceleration. A communication deadline should relate to how long a message remains useful. A security verification deadline should relate to whether authentication or IDS checks finish before the receiver uses the message.

### 10. Common Architectures Used Across These Experiments

Most experiments in this lab can be understood using one of three architecture patterns.

Control-loop pattern:

```
Sensor -> Controller Task -> Actuator -> Plant / Vehicle -> Sensor
```

Communication-loop pattern:

```
Publisher / Sender -> Network Medium -> Receiver / Subscriber -> Application Decision
```

Security-monitoring pattern:

```
Message Source -> Security Check -> IDS / Risk Logic -> Accept, Reject, or Alert
```

The control-loop pattern focuses on WCET, response time, and deadline satisfaction. The communication-loop pattern focuses on latency, jitter, throughput, packet loss, and deterministic delivery. The security-monitoring pattern focuses on integrity, authentication, replay resistance, anomaly detection, and risk reduction. Autonomous systems usually combine all three patterns, which is why timing and security cannot be treated as separate afterthoughts.

### 11. Common Mistakes to Avoid in Lab Answers

Do not say "real-time means fast." Say "real-time means deadline-bound." Do not use average execution time as a substitute for WCET in hard real-time analysis. Do not conclude that high throughput guarantees good real-time performance. Do not claim a security mechanism provides authentication unless the mechanism actually proves sender identity. Do not claim a physical simulator or broker was used if the notebook uses a Python fallback. Clear assumptions make the lab record more credible.

### Core References for These Notes

- Python timing functions such as `perf_counter()` and monotonic clocks are documented by the official Python `time` module documentation: https://docs.python.org/3/library/time.html
- IEEE 802.1 Time-Sensitive Networking is the IEEE working-group area for time-sensitive network behavior: https://1.ieee802.org/tsn/
- SUMO official documentation describes traffic simulation concepts used in V2V mobility experiments: https://sumo.dlr.de/docs/
- MQTT is an OASIS publish-subscribe messaging standard for IoT telemetry: https://docs.oasis-open.org/mqtt/mqtt/v5.0/mqtt-v5.0.html
- NIST FIPS 180-4 specifies SHA-256 as part of the Secure Hash Standard: https://csrc.nist.gov/pubs/fips/180-4/upd1/final
- NIST SP 800-30 Rev. 1 provides risk-assessment guidance: https://csrc.nist.gov/pubs/sp/800/30/r1/final

## Extended Experiment Notes and Case Studies

### Experiment Focus

This experiment studies throughput and response time. Throughput tells how much data or how many requests are completed per second. Response time tells how long each request waits from arrival to completion. Real-time systems need both sufficient throughput and bounded response time.

### Experiment Architecture

```
Request Source
  -> Arrival Queue
  -> Link / Processor / Server
  -> Completed Request Log
  -> Throughput and Response-Time Analysis
```

When arrival rate approaches service capacity, queue length grows. This increases response time even if the server is busy and throughput looks high.

### Detailed Formula Set

```
throughput = completed_work / observation_time
response_time = completion_time - arrival_time
utilization = arrival_rate / service_rate
average_queue_delay = total_waiting_time / number_of_requests
```

For a simplified single-server queue:

```
expected_response_time = 1 / (service_rate - arrival_rate)
```

This formula is only an analytical model, but it shows why response time grows sharply near overload.

### Case Study 1 - Sensor Fusion Gateway

Camera, lidar, radar, and GPS data arrive at a gateway. If large camera packets fill the queue, small safety messages may be delayed. Priority scheduling or separate queues may be required.

### Case Study 2 - Roadside Edge Server

An edge server receives hazard reports from vehicles near an intersection. During congestion, offered load rises. The server must preserve bounded response time for urgent messages, not merely maximize total processed messages.

### Case Study 3 - Fleet Telemetry Upload

Telemetry upload can be high-throughput but low-criticality. If telemetry competes with safety messages, rate limiting or traffic shaping is needed.

### Lab Record Guidance

Report offered load, completed work, throughput, response times, queue behavior, and any overload point. State whether the measured response time satisfies the application deadline.

## Architecture

```text
Packet Workload
  |-- packet size
  |-- link rate
  |-- queue delay
  |-- processing delay
          |
          v
Transmission Model
  |-- transmission time = packet bits / link rate
          |
          v
Response-Time Model
  |-- response time = transmission + queueing + processing
          |
          v
Throughput Calculator
  |-- total delivered bits / elapsed time
```

The simulation separates link capacity from queueing delay because congestion often shows up first as response-time growth.

## Formulas and Required Theory

\[
\text{transmission time} = \frac{\text{packet size in bits}}{\text{link rate in bits/s}}
\]

\[
\text{response time} = T_{transmission} + T_{queueing} + T_{processing}
\]

\[
\text{throughput} = \frac{\text{total delivered bits}}{\text{total elapsed time}}
\]

When offered load increases, queueing delay usually grows faster than raw transmission time. This is why autonomous systems need both bandwidth planning and latency monitoring.

## In-Lab Method

1. Generate packets with different payload sizes.
2. Compute transmission time from packet size and link rate.
3. Add queueing and processing delays.
4. Measure total delivered bits and total elapsed time.
5. Report throughput, mean response time, and maximum response time.

In [1]:
import random
import statistics

rng = random.Random(341403)
packets = []
clock = 0.0
for i in range(40):
    size_bytes = rng.choice([128, 256, 512, 1024, 1400])
    link_mbps = 12
    tx_ms = (size_bytes * 8) / (link_mbps * 1000)
    queue_ms = rng.uniform(0.2, 3.8)
    processing_ms = rng.uniform(0.4, 2.5)
    response_ms = tx_ms + queue_ms + processing_ms
    clock += response_ms
    packets.append((i + 1, size_bytes, response_ms))

total_bits = sum(p[1] * 8 for p in packets)
duration_s = clock / 1000
throughput_mbps = total_bits / duration_s / 1_000_000
responses = [p[2] for p in packets]

print("EXP 3 - IN-LAB RESULT")
print(f"Packets analysed      : {len(packets)}")
print(f"Total payload         : {total_bits / 8:.0f} bytes")
print(f"Measured throughput   : {throughput_mbps:.3f} Mbps")
print(f"Mean response time    : {statistics.mean(responses):.2f} ms")
print(f"Max response time     : {max(responses):.2f} ms")
print("First five packets:", packets[:5])

EXP 3 - IN-LAB RESULT
Packets analysed      : 40
Total payload         : 31264 bytes
Measured throughput   : 1.541 Mbps
Mean response time    : 4.06 ms
Max response time     : 6.83 ms
First five packets: [(1, 512, 2.220407117623182), (2, 128, 3.795172099694435), (3, 1024, 2.431030740592837), (4, 1024, 5.554517512164553), (5, 1400, 2.8858043128709827)]


## Post-Lab Method

The post-lab cell repeats the experiment under different load multipliers. It shows how response time increases when the communication system becomes busier.

In [2]:
import random
import statistics

def trial(load, seed=341430):
    rng = random.Random(seed)
    responses = []
    bits = 0
    elapsed = 0.0
    for _ in range(100):
        size = rng.choice([256, 512, 1024, 1400])
        link_mbps = 12
        tx = (size * 8) / (link_mbps * 1000)
        queue = rng.uniform(0.3, 3.0) * load
        proc = rng.uniform(0.4, 2.2)
        rt = tx + queue + proc
        responses.append(rt)
        elapsed += rt
        bits += size * 8
    return bits / (elapsed / 1000) / 1_000_000, statistics.mean(responses), max(responses)

print("EXP 3 - POST-LAB LOAD EFFECT")
print(f"{'Load':>6} {'Throughput Mbps':>16} {'Mean RT ms':>12} {'Max RT ms':>10}")
for load in [0.5, 0.8, 1.0, 1.3, 1.6]:
    throughput, mean_rt, max_rt = trial(load)
    print(f"{load:6.1f} {throughput:16.3f} {mean_rt:12.2f} {max_rt:10.2f}")

EXP 3 - POST-LAB LOAD EFFECT
  Load  Throughput Mbps   Mean RT ms  Max RT ms
   0.5            2.634         2.66       4.23
   0.8            2.235         3.13       5.05
   1.0            2.030         3.45       5.60
   1.3            1.785         3.92       6.43
   1.6            1.592         4.39       7.25


## What to Write in the Lab Record

- Record throughput in Mbps.
- Record mean and maximum response time.
- Explain why maximum response time matters for autonomous control.
- Compare how load changes throughput and delay.

## References

- Python `time` module documentation: https://docs.python.org/3/library/time.html
- Python `statistics` module documentation: https://docs.python.org/3/library/statistics.html
- Python `random` module documentation: https://docs.python.org/3/library/random.html